# MosAIc: comparing baselines

The baseline is MosAIc's starting forecast, made from prices alone before any filings or news are read: the probability that the stock falls a lot, stays roughly flat or rises a lot over the next 21 trading days. Every ledger line is a shift away from it, so if the baseline is off, the whole forecast is off.

**Why this notebook.** In the Tree of Thought validation run, two days after each results announcement, the baseline scored worse than a fixed 31% / 38% / 31% guess. The results-day jump had just happened, the baseline counted it as "the stock is very jumpy right now" and predicted big moves, but the month after results is usually calm (11 of 21 months were flat).

**The versions compared.** Each adds one idea to the one before, so each step's effect can be seen on its own:

| Version | What it does |
| --- | --- |
| A | current baseline: recent daily moves, the last few days counting most, plus the typical results-day jump if results are expected within the month |
| B | A, but results-day moves are left out of "recent moves"; results count only through the results-day jump term |
| C | B mixed half and half with the past year's moves (without results days), so a few unusual weeks matter less |
| D | C adjusted by how calm or jumpy the month usually is at this point after results, learned from past years of all 30 stocks |
| Fixed guess | 30.85% / 38.3% / 30.85% every time, the probabilities if the coming month moves like an average month of the past year |

**Why this can be tested on thousands of forecasts.** The baseline needs no language model, only arithmetic on prices. So all 30 stocks are tested every week from 2016, about 16,000 forecasts, against what actually happened. Version D only learns from windows that ended before each forecast date, so every forecast is a fair out-of-sample test.

**Later:** once the FirstRate option data is loaded, the same forecasts can also be scored against the option-implied probabilities, the main metric of the design.

In [ ]:
!pip install -q yfinance scipy

## 1. Settings

In [ ]:
USER_AGENT = "Your Name your.email@example.com"     # required by the SEC

TICKERS = ["AAPL", "MSFT", "NVDA", "INTC", "GOOGL", "META", "NFLX", "AMZN", "TSLA", "NKE",
           "JPM", "GS", "V", "JNJ", "UNH", "LLY", "PG", "KO", "WMT", "XOM", "CVX",
           "CAT", "BA", "LMT", "NEE", "DUK", "PLD", "AMT", "LIN", "FCX"]

PRICE_START = "2012-01-01"          # price history to download
TEST_START, TEST_END = "2016-01-04", "2026-09-25"   # weekly forecast dates (Mondays)

# Forecast definition (Checkpoint 1.1)
HORIZON = 21
SIGMA_LOOKBACK = 252
BUCKET_K = 0.5

# Baseline pieces
EWMA_DECAY = 0.94                   # recent days weigh more (version A, as in the notebooks so far)
BLEND_RECENT = 0.5                  # version C: weight on recent moves; the rest on the past year
EARNINGS_GAP_DAYS = 91              # next results expected this long after the last
EARNINGS_WINDOW_DAYS = 30           # "within the month" for the results-day jump term
AFTER_CLOSE_UTC_HOUR = 20           # results filed at or after this hour (UTC) move the stock the next day

# Version D: calm-down profile, by trading days since the last results
CALM_BINS = [(1, 5), (6, 15), (16, 30), (31, 45), (46, 63), (64, 999)]
CALM_MIN_WINDOWS = 50               # fewer past windows than this: no adjustment

FIXED_GUESS = {"fall": 30.85, "flat": 38.3, "gain": 30.85}
N_BOOT = 1000

In [ ]:
import math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
from scipy.stats import norm

OUTCOMES = ("fall", "flat", "gain")
VERSIONS = ("A", "B", "C", "D", "Fixed guess")

## 2. Data: results dates and prices

Results dates come from the SEC's list of each company's filings: an 8-K with item 2.02 is a results announcement. Only the list is downloaded, not the documents, so this takes a minute. Prices come from Yahoo Finance (adjusted closes).

In [ ]:
import json, re, time, requests

HEADERS = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate"}

def sec_get(url):
    time.sleep(0.15)
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r

def results_times(cik):
    """Acceptance times (UTC) of every results 8-K (item 2.02), over the full filing history."""
    sub = sec_get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json").json()
    blocks = [sub["filings"]["recent"]] + [sec_get(f"https://data.sec.gov/submissions/{p['name']}").json()
                                           for p in sub["filings"].get("files", [])]
    out = []
    for b in blocks:
        for form, items, acc in zip(b["form"], b.get("items", [""] * len(b["form"])), b["acceptanceDateTime"]):
            if form == "8-K" and "2.02" in str(items):
                out.append(datetime.fromisoformat(acc.replace("Z", "+00:00")).astimezone(timezone.utc))
    return sorted(out)

cik_map = {v["ticker"].upper(): int(v["cik_str"])
           for v in sec_get("https://www.sec.gov/files/company_tickers.json").json().values()}
RESULTS = {t: results_times(cik_map[t]) for t in TICKERS}
for t in TICKERS:
    print(f"{t}: {len(RESULTS[t])} results filings since {RESULTS[t][0].date() if RESULTS[t] else '-'}")

In [ ]:
import yfinance as yf
PRICES = {}
for t in TICKERS:
    px = yf.download(t, start=PRICE_START, progress=False, auto_adjust=True)["Close"]
    PRICES[t] = px.squeeze().dropna()
print({t: len(p) for t, p in PRICES.items()})